# 16.07 - Test-Time Augmentation

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** A mini experiment that adds horizontal-flip test-time augmentation (TTA), measures validation accuracy and runtime, and reports the speed/quality tradeoff.

Training augmentation teaches invariance; test-time augmentation asks the trained model to vote across a small set of label-preserving views. Today you will implement horizontal-flip TTA, average probabilities correctly, and decide whether its measured benefit justifies its inference cost.


## Core Ideas

### 1. TTA is an inference ensemble

Given an image `x` and a label-preserving transform `flip(x)`, run the same model on both views and average their class probabilities. The model parameters do not change. With two views, inference usually requires about twice as many forward-pass samples.

### 2. Average comparable outputs

This notebook averages probabilities after softmax: `0.5 * (p(x) + p(flip(x)))`. Probability averaging is easy to interpret and remains normalized. Logit averaging is another valid design, but it is a different experiment and can produce different predictions.

### 3. Transforms must preserve labels

A horizontal flip is often valid for generic object categories, but not for text, traffic direction, handedness, asymmetrical medical findings, or labels such as `left-facing`. Confirm domain semantics before using TTA.

### 4. Keep evaluation deterministic

Use `model.eval()` and `torch.no_grad()`. Do not use random training transforms during validation. Apply the exact same preprocessing and class mapping to every TTA view.

### 5. Measure the tradeoff

Record accuracy (or the contest metric), elapsed time, and the overhead ratio on the same examples and device. Synchronize the accelerator before and after timing when using CUDA. Tiny CPU timings are noisy, so repeat measurements in production benchmarks.


In [ ]:
import time
import numpy as np
import torch
from torch import nn

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

CLASS_NAMES = ["circle", "square", "triangle"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Prepared Image Data and Classifier

The prepared classifier intentionally reacts to whether a bright nuisance stripe appears on the left or right. The real class is encoded by the dominant color channel. Averaging original and flipped probabilities cancels much of this orientation shortcut. The classifier and data generator are provided so the exercises stay focused on inference and measurement.

**Return structure — `make_day16_validation_data(...)`:** a 2-tuple `(images, labels)` where `images` is a CPU `torch.Tensor` with dtype `float32` and shape `[N, 3, H, W]`, and `labels` is a CPU `torch.Tensor` with dtype `torch.long` and shape `[N]`.

**Return structure — `FlipBiasedClassifier(...)`:** a `FlipBiasedClassifier` module. Calling it as `model(images)` invokes `forward` and returns a floating-point logits `torch.Tensor` on the input device with shape `[N, 3]`.


In [ ]:
def make_day16_validation_data(n_samples=24, image_size=24):
    images = torch.full((n_samples, 3, image_size, image_size), 0.05, dtype=torch.float32)
    labels = torch.tensor([i % 3 for i in range(n_samples)], dtype=torch.long)
    stripe_width = image_size // 4
    for index, label in enumerate(labels.tolist()):
        images[index, label] = 0.55
        if index % 2 == 0:
            images[index, :, :, :stripe_width] = 0.90
        else:
            images[index, :, :, -stripe_width:] = 0.90
    return images, labels


class FlipBiasedClassifier(nn.Module):
    def __init__(self, bias_strength=1.1):
        super().__init__()
        self.bias_strength = float(bias_strength)

    def forward(self, images):
        color_evidence = images.mean(dim=(2, 3))
        width = images.shape[-1]
        left = images[:, :, :, : width // 4].mean(dim=(1, 2, 3))
        right = images[:, :, :, -width // 4 :].mean(dim=(1, 2, 3))
        direction = self.bias_strength * (left - right)
        shortcut = torch.stack([direction, -direction, torch.zeros_like(direction)], dim=1)
        return 4.0 * color_evidence + 4.0 * shortcut


val_images, val_labels = make_day16_validation_data()
model = FlipBiasedClassifier().to(DEVICE)
print("images:", tuple(val_images.shape), val_images.dtype)
print("labels:", tuple(val_labels.shape), val_labels.dtype)
print("device:", DEVICE)
print("label mapping:", dict(enumerate(CLASS_NAMES)))


## Exercise 16-A: Horizontal-Flip a Batch

Implement `horizontal_flip_batch(images)` for a `[N, C, H, W]` tensor. Reverse only the width dimension and validate the input rank.

**Return structure:** one `torch.Tensor` with the same shape `[N, C, H, W]`, dtype, and device as `images`.


In [ ]:
def horizontal_flip_batch(images):
    if images.ndim != 4:
        raise ValueError("images must have shape [N, C, H, W]")
    return torch.flip(images, dims=[-1])


flipped_preview = horizontal_flip_batch(val_images[:2])
print("flipped batch:", tuple(flipped_preview.shape), flipped_preview.dtype, flipped_preview.device)


## Exercise 16-B: Predict Probabilities in Batches

Implement `predict_probabilities(model, images, batch_size=8)`. Temporarily use evaluation mode, disable gradient tracking, move each mini-batch to the model device, apply softmax over classes, and restore the model's original train/eval state before returning.

**Return structure:** one detached CPU `torch.Tensor` with floating-point dtype and shape `[N, C]`. Each row contains class probabilities and sums to `1`.


In [ ]:
def predict_probabilities(model, images, batch_size=8):
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")
    parameter = next(model.parameters(), None)
    model_device = parameter.device if parameter is not None else images.device
    was_training = model.training
    model.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(images), batch_size):
            batch = images[start : start + batch_size].to(model_device)
            outputs.append(torch.softmax(model(batch), dim=1).cpu())
    model.train(was_training)
    return torch.cat(outputs, dim=0)


baseline_probabilities = predict_probabilities(model, val_images)
print("probabilities:", tuple(baseline_probabilities.shape), baseline_probabilities.device)


## Exercise 16-C: Average Original and Flipped Predictions

Implement `horizontal_flip_tta(model, images, batch_size=8)`. Predict both original and horizontally flipped views and average them. Do not flip labels or reorder examples.

**Return structure:** one detached CPU `torch.Tensor` with floating-point dtype and shape `[N, C]`. Row `i` is the arithmetic mean of the original and flipped probability vectors for example `i` and sums to `1`.


In [ ]:
def horizontal_flip_tta(model, images, batch_size=8):
    original_probabilities = predict_probabilities(model, images, batch_size=batch_size)
    flipped_images = horizontal_flip_batch(images)
    flipped_probabilities = predict_probabilities(model, flipped_images, batch_size=batch_size)
    return 0.5 * (original_probabilities + flipped_probabilities)


tta_probabilities = horizontal_flip_tta(model, val_images)
print("TTA probability sums:", tta_probabilities.sum(dim=1)[:3])


## Exercise 16-D: Evaluate Predictions

Implement `classification_summary(probabilities, labels)`. Confidence is the maximum class probability per sample.

**Return structure:** a dictionary with the exact schema:

- `accuracy`: Python `float` in `[0, 1]`
- `predictions`: CPU `torch.Tensor`, dtype `torch.long`, shape `[N]`
- `confidence`: CPU floating-point `torch.Tensor`, shape `[N]`


In [ ]:
def classification_summary(probabilities, labels):
    if probabilities.ndim != 2:
        raise ValueError("probabilities must have shape [N, C]")
    labels_cpu = labels.detach().cpu()
    if labels_cpu.ndim != 1 or len(labels_cpu) != len(probabilities):
        raise ValueError("labels must have shape [N]")
    confidence, predictions = probabilities.detach().cpu().max(dim=1)
    accuracy = float((predictions == labels_cpu).float().mean())
    return {
        "accuracy": accuracy,
        "predictions": predictions.to(torch.long),
        "confidence": confidence,
    }


print("baseline:", classification_summary(baseline_probabilities, val_labels)["accuracy"])
print("flip TTA:", classification_summary(tta_probabilities, val_labels)["accuracy"])


## Exercise 16-E: Measure the TTA Experiment

Implement `run_tta_experiment(model, images, labels, batch_size=8)`. Time the baseline prediction and the two-view TTA prediction separately, and synchronize CUDA around each timing block when needed. Interpret the metric change together with the extra runtime.

**Return structure:** a dictionary with the exact schema:

- `baseline`: summary dictionary from `classification_summary(...)`
- `tta`: summary dictionary from `classification_summary(...)`
- `baseline_seconds`: Python `float` greater than `0`
- `tta_seconds`: Python `float` greater than `0`
- `overhead_ratio`: Python `float`, computed as `tta_seconds / baseline_seconds`


In [ ]:
def run_tta_experiment(model, images, labels, batch_size=8):
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    baseline_start = time.perf_counter()
    baseline_probabilities = predict_probabilities(model, images, batch_size=batch_size)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    baseline_seconds = time.perf_counter() - baseline_start

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    tta_start = time.perf_counter()
    tta_probabilities = horizontal_flip_tta(model, images, batch_size=batch_size)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    tta_seconds = time.perf_counter() - tta_start

    return {
        "baseline": classification_summary(baseline_probabilities, labels),
        "tta": classification_summary(tta_probabilities, labels),
        "baseline_seconds": baseline_seconds,
        "tta_seconds": tta_seconds,
        "overhead_ratio": tta_seconds / max(baseline_seconds, 1e-12),
    }


experiment = run_tta_experiment(model, val_images, val_labels)
print(f"baseline accuracy: {experiment['baseline']['accuracy']:.3f}")
print(f"TTA accuracy:      {experiment['tta']['accuracy']:.3f}")
print(f"baseline time:     {experiment['baseline_seconds'] * 1000:.3f} ms")
print(f"TTA time:          {experiment['tta_seconds'] * 1000:.3f} ms")
print(f"runtime overhead:  {experiment['overhead_ratio']:.2f}x")


## Test Cases

Run this cell after completing all TODO cells. The tests verify width-only flipping, the double-flip identity, probability normalization, shape/dtype/device contracts, model-mode restoration, TTA averaging, metric summaries, and timing outputs. A correct implementation prints `Day 16 tests passed`.

**Return structure — `run_day16_tests()`:** `None`. Success is communicated by completed assertions and the printed confirmation.


In [ ]:
def run_day16_tests():
    sample = torch.arange(2 * 3 * 4 * 5, dtype=torch.float32).reshape(2, 3, 4, 5)
    flipped = horizontal_flip_batch(sample)
    assert flipped.shape == sample.shape
    assert flipped.dtype == sample.dtype
    assert flipped.device == sample.device
    assert torch.equal(flipped[:, :, :, 0], sample[:, :, :, -1])
    assert torch.equal(horizontal_flip_batch(flipped), sample)

    model.train()
    baseline = predict_probabilities(model, val_images, batch_size=5)
    assert model.training, "predict_probabilities must restore the original model mode"
    assert baseline.shape == (len(val_images), len(CLASS_NAMES))
    assert baseline.dtype == torch.float32
    assert baseline.device.type == "cpu"
    assert torch.allclose(baseline.sum(dim=1), torch.ones(len(val_images)), atol=1e-6)

    flipped_probabilities = predict_probabilities(model, horizontal_flip_batch(val_images), batch_size=5)
    tta = horizontal_flip_tta(model, val_images, batch_size=5)
    assert torch.allclose(tta, 0.5 * (baseline + flipped_probabilities), atol=1e-6)
    assert torch.allclose(tta.sum(dim=1), torch.ones(len(val_images)), atol=1e-6)

    baseline_summary = classification_summary(baseline, val_labels)
    tta_summary = classification_summary(tta, val_labels)
    assert baseline_summary["predictions"].dtype == torch.long
    assert baseline_summary["predictions"].device.type == "cpu"
    assert baseline_summary["confidence"].shape == val_labels.shape
    assert 0.0 <= baseline_summary["accuracy"] <= 1.0
    assert tta_summary["accuracy"] >= baseline_summary["accuracy"]

    result = run_tta_experiment(model, val_images, val_labels, batch_size=6)
    assert set(result) == {
        "baseline", "tta", "baseline_seconds", "tta_seconds", "overhead_ratio"
    }
    assert result["baseline_seconds"] > 0.0
    assert result["tta_seconds"] > 0.0
    assert result["overhead_ratio"] > 0.0
    print("Day 16 tests passed")


run_day16_tests()


## Day 16 Checklist

- [ ] I can explain TTA as an inference-time ensemble without parameter updates.
- [ ] I confirm that horizontal flips preserve the task's label semantics.
- [ ] I use `model.eval()` and `torch.no_grad()` during prediction.
- [ ] I average probabilities for corresponding examples and verify rows still sum to one.
- [ ] I preserve tensor shape, dtype, device handling, and label mapping.
- [ ] I compare the validation metric and measured runtime before deciding to deploy TTA.
